In [1]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# For simple label encoding to one-hot
from sklearn.preprocessing import OneHotEncoder

from src.backend import BackendPolicy, backend
from src.layers.activation import ELU
from src.layers.activation.softmax import Softmax
from src.layers.dense import Dense
from src.loss.cross_entropy import CrossEntropy
from src.models.sequential import Sequential
from src.optimizer.adam import Adam


In [ ]:
from sklearn.datasets import load_digits
from sklearn.datasets import fetch_openml
digits = load_digits()
x, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
x = x.astype(np.float32) / 255.0  # Scale to [0, 1]
y = y.astype(int)
use_gpu = True

x_train, x_test, y_train_labels, y_test_labels = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
dataset_bytes = x_train.nbytes + y_train_labels.nbytes

encoder = OneHotEncoder(sparse_output=False, dtype=np.float32)
y_train = encoder.fit_transform(y_train_labels.reshape(-1, 1))
y_test = encoder.transform(y_test_labels.reshape(-1, 1))

2026-03-07 20:19:11,962 | INFO | src.backend | GPU requested. CuPy available: True.
2026-03-07 20:19:11,963 | INFO | src.backend | Using backend: cupy


In [ ]:

import time


def benchmark(epochs=20, verbose=10, use_gpu=True):
    backend.configure(
    BackendPolicy(use_gpu=use_gpu,min_gpu_bytes=1),
    dataset_bytes=dataset_bytes,

    )

    model = Sequential(
    [
        Dense(units=128, name="dense_1"),
        ELU(name="elu_1"),
        Dense(units=16, name="dense_2"),
        ELU(name="elu_2"),
        Dense(units=10, name="dense_out"),
        Softmax(name="softmax_out"),
    ]
)

    model.compile(loss=CrossEntropy(), optimizer=Adam(learning_rate=1e-3))
    start = time.time()

    history = model.fit(x_train, y_train, epochs=epochs, verbose=verbose)
    end = time.time()
    train_probs = model.predict(x_train)
    test_probs = model.predict(x_test)

    train_preds = np.argmax(train_probs, axis=1)
    test_preds = np.argmax(test_probs, axis=1)

    train_acc = accuracy_score(y_train_labels, train_preds)
    test_acc = accuracy_score(y_test_labels, test_preds)
    test_precision = precision_score(y_test_labels, test_preds, average="weighted", zero_division=0)
    test_recall = recall_score(y_test_labels, test_preds, average="weighted", zero_division=0)
    test_f1 = f1_score(y_test_labels, test_preds, average="weighted", zero_division=0)

    print(f"Final train loss: {history[-1]:.6f}")
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Test accuracy:  {test_acc:.4f}")
    print(f"Test precision (weighted): {test_precision:.4f}")
    print(f"Test recall (weighted):    {test_recall:.4f}")
    print(f"Test F1 (weighted):        {test_f1:.4f}")

    sample_idx = np.arange(10)
    print("Predictions:", test_preds[sample_idx])
    print("Ground truth:", y_test_labels[sample_idx])
    return (end - start)

In [ ]:
epochs = 100
verbose = 25
no_gpu = benchmark(use_gpu=False, epochs=epochs, verbose=verbose)
gpu = benchmark(use_gpu=True, epochs=epochs, verbose=verbose)
print(f"Time without GPU: {no_gpu:.2f} seconds")
print(f"Time with GPU:    {gpu:.2f} seconds")

2026-03-07 20:23:07,413 | INFO | src.backend | Using backend: numpy
Epoch 1/20 - Loss: 2.498800
Epoch 11/20 - Loss: 1.211028
Epoch 20/20 - Loss: 0.836528
Final train loss: 0.836528
Train accuracy: 0.7671
Test accuracy:  0.7722
Test precision (weighted): 0.7787
Test recall (weighted):    0.7722
Test F1 (weighted):        0.7456

Confusion matrix:
[[1278    0   28    7    4   17   23    1   22    1]
 [   1 1485   13   10    2    3    9    3   49    0]
 [  31   33 1134   79   13    4   52   15   28    9]
 [  11   23   64 1196    1   55   15   29   32    2]
 [   9   15   27    1 1186   20   37    6   60    4]
 [  45   20   23  102   58  797   31   27  156    4]
 [  18   13   11    0    7   30 1283    1    6    6]
 [  14   38   25    8   17    1    4 1313   29   10]
 [  27   84   34  109   19   62   11   10 1004    5]
 [  31   26   16   13  465   34    2  275  394  135]]

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.93      0.9